# Phase 3 : 데이터 분석(EDA)

# [Step 1] 저장된 파일 로드 및 기본 구조 확인

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. 머신러닝용 정형 피처 로드
pdf = pd.read_parquet('./데이터/ml_tabular_features.parquet')

# 2. 딥러닝용 시퀀스 텐서 및 타겟 레이블 로드
X_seq = np.load('./데이터/dl_sequence_tensor.npy', allow_pickle=True)
y = np.load('./데이터/churn_labels.npy', allow_pickle=True)

print('--- [데이터 로드 완료] ---')
print(f'1. ML 정형 피처 형태 (Shape): {pdf.shape}')
print(f'2. DL 시퀀스 텐서 형태 (Shape): {X_seq.shape}')
print(f'3. 타겟 레이블 형태 (Shape): {y.shape}')

print('\n[정형 피처 컬럼 목록]')
print(pdf.columns.tolist())

--- [데이터 로드 완료] ---
1. ML 정형 피처 형태 (Shape): (22278, 14)
2. DL 시퀀스 텐서 형태 (Shape): (22278, 50)
3. 타겟 레이블 형태 (Shape): (22278,)

[정형 피처 컬럼 목록]
['userId', 'is_churn', 'gender', 'level', 'total_interactions', 'total_songs_played', 'total_thumbs_up', 'total_thumbs_down', 'total_playlist_adds', 'total_errors', 'total_help', 'total_sessions', 'total_listening_time_sec', 'session_item_sequence']


# [Step 2] 타겟(is_churn) 클래스 불균형 분석

In [2]:
# 이탈 여부 분포 계산
churn_counts = (
    pdf['is_churn']
    .value_counts()
    .reset_index(name='count')
    .rename(columns={'is_churn': 'churn'})
)
churn_counts['churn_label'] = churn_counts['churn'].map(
    {0: '유지 (0)', 1: '이탈 (1)'}
)
churn_counts['ratio'] = (
    churn_counts['count'] / churn_counts['count'].sum()
) * 100

# Plotly 바 차트
fig = px.bar(
    churn_counts,
    x='churn_label',
    y='count',
    text=churn_counts['ratio'].round(2).astype(str) + '%',
    color='churn_label',
    title='<b>사용자 이탈(Churn) 클래스 분포</b>',
    labels={'churn_label': '이탈 여부', 'count': '사용자 수'},
    color_discrete_sequence=['#636EFA', '#EF553B'],
)

fig.update_traces(textposition='outside')
fig.update_layout(
    template='plotly_white', showlegend=False, yaxis_range=[0, churn_counts['count'].max() * 1.15]
)
fig.show()

# [Step 3] 정형 피처 기반 이탈 그룹 비교 EDA

In [3]:
target_features = [
    'total_songs_played',
    'total_thumbs_down',
    'total_errors',
    'total_sessions',
]

# 서브플롯 생성 (2행 2열)
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        '총 재생 곡 수 (total_songs_played)',
        '싫어요 횟수 (total_thumbs_down)',
        '에러 발생 횟수 (total_errors)',
        '총 접속 세션 수 (total_sessions)',
    ],
)

row_col_indices = [(1, 1), (1, 2), (2, 1), (2, 2)]

for idx, feature in enumerate(target_features):
  r, c = row_col_indices[idx]
  if feature in pdf.columns:
    fig.add_trace(
        go.Box(
            x=pdf['is_churn'].astype(str),
            y=pdf[feature],
            name=feature,
            boxpoints='outliers',  # 이상치 표시
            marker_color='#EF553B' if idx % 2 == 1 else '#636EFA',
        ),
        row=r,
        col=c,
    )

fig.update_layout(
    template='plotly_white',
    title_text='<b>이탈 여부(0 vs 1)에 따른 주요 정형 피처 비교</b>',
    showlegend=False,
    height=750,
)
fig.show()

In [4]:
# 2. 수치형 피처만 자동으로 추출
# (userId, gender, level, last_page, session_item_sequence 등 범주형 및 시퀀스 컬럼 제외)
exclude_cols = [
    'userId',
    'gender',
    'level',
    'last_page',
    'session_item_sequence',
]
numeric_pdf = pdf.select_dtypes(include=[np.number]).drop(
    columns=[col for col in exclude_cols if col in pdf.columns], errors='ignore'
)

# 3. 피처 간 상관계수(Correlation Matrix) 계산 (피어슨 상관계수 기준)
corr_matrix = numeric_pdf.corr(method='pearson')

# 4. Plotly를 활용한 인터랙티브 히트맵 시각화
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',  # 각 셀에 소수점 둘째 자리까지 상관계수 표시
    color_continuous_scale='RdBu_r',  # 양의 상관관계는 빨강, 음의 상관관계는 파랑
    zmin=-1,
    zmax=1,
    aspect='auto',  # 데이터 크기에 맞춰 비율 자동 조절
    title='<b>정형 피처 간 상관관계 히트맵 (Correlation Heatmap)</b>',
)

# 레이아웃 디자인 설정
fig.update_layout(
    template='plotly_white',
    width=900,
    height=800,
    xaxis_title='<b>Features</b>',
    yaxis_title='<b>Features</b>',
)

# 시각화 출력
fig.show()

# [Step 4] 딥러닝 시퀀스 데이터(X_seq) 구조 및 길이 검증

In [5]:
print(f'시퀀스 텐서 총 샘플 수 (유저 수): {X_seq.shape[0]}')
print(f'각 유저별 시퀀스 최대 길이 (Max Seq Len): {X_seq.shape[1]}')
if len(X_seq.shape) > 2:
  print(f'시퀀스 내부 피처 차원 (Embedding/Feature Dim): {X_seq.shape[2]}')

# (선택) 만약 X_seq가 정수 인덱스나 패딩 값(예: 0)을 포함하는 경우, 유효한 시퀀스 길이 분포 확인 가능
if X_seq.ndim == 2:
  # 예: 0이 아닌 유효한 이벤트 개수 계산
  valid_lengths = np.sum(X_seq != 0, axis=1)

  fig = px.histogram(
      x=valid_lengths,
      nbins=30,
      title='<b>유저별 유효 시퀀스 길이 분포 (Valid Sequence Lengths)</b>',
      labels={'x': '이벤트 수', 'y': '유저 수'},
      color_discrete_sequence=['#00CC96'],
  )
  fig.update_layout(template='plotly_white')
  fig.show()

시퀀스 텐서 총 샘플 수 (유저 수): 22278
각 유저별 시퀀스 최대 길이 (Max Seq Len): 50
